In [253]:
import os
import time
import random
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors
from unittest.mock import MagicMock
from dataclasses import dataclass, field



In [254]:
load_dotenv()
client = genai.Client()

### Gemini api call with streaming enabled

In [255]:
@dataclass
class API_Configs:
    model_config = types.GenerateContentConfig(system_instruction="do not provide code",max_output_tokens=4096)
    
    model_tiers = {
    "high": {"model_name": "gemini-3.5-flash"},
    "medium": {"model_name": "gemini-2.5-flash"},
    "low": {"model_name": "gemini-3.1-flash-lite"}
}

In [256]:
@dataclass
class Retry_Config:
    max_attempts: int = 3
    base_delay: float = 1.0
    max_delay: float = 30.0

    retry_status_codes = (429, 500, 503, 504)

In [257]:
model_priority = "high"

In [258]:
tier = API_Configs.model_tiers.get(model_priority.lower(), API_Configs.model_tiers["medium"])
model_name = tier["model_name"]

### api call with retry error handling, exponential backoff, and streaming

In [259]:
def execute_prompt(model_name, prompt):
    for attempt in range (1, Retry_Config.max_attempts + 1):
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print(f"Sending initial request...")

            stream = client.models.generate_content_stream(model = model_name, contents = prompt)

            for chunk in stream:
                yield chunk

            return 
        
        except errors.APIError as e:
            if e.code in Retry_Config.retry_status_codes:
                if attempt == Retry_Config.max_attempts:
                    print("Max attempts reached.")
                    raise e
            
                new_delay = random.uniform(0.5, 1.5) * Retry_Config.base_delay * (2 ** (attempt - 1))
                clamped_delay = min(new_delay, Retry_Config.max_delay)

                print(f" Code {e.code}. Backing off. Waiting {clamped_delay:.2f}s...")
                time.sleep(clamped_delay)
                continue

            else:
                status_reason = getattr(e, "status", "UNKNOWN")
                
                if e.code == 404 or status_reason == "NOT_FOUND":
                    print(f"Model Not Found [404]: '{model_name}' does not exist or is deprecated.")
                    
                elif e.code in (401, 403) or status_reason == "PERMISSION_DENIED":
                    print(f"Authentication Failed [{e.code}]: Check your API key, project quotas, or permissions.")
                    
                elif e.code == 400 or status_reason == "INVALID_ARGUMENT":
                    print(f"Invalid Argument [400]: Malformed structure or prompt content limits exceeded.")
                    
                else:
                    print(f"Non-retryable error [{status_reason}] ({e.code}): {e.message}")
                    
                raise e

In [260]:
response_stream = execute_prompt(model_name = model_name, prompt = "Say hello in 10 random languages")

for chunk in response_stream:
    print(chunk.text, end="", flush=True)

Sending initial request...
Here is "hello" in 10 different languages from around the world:

1. **Japanese**: こんにちは (Konnichiwa)
2. **Swahili**: Jambo
3. **German**: Hallo
4. **Hindi**: नमस्ते (Namaste)
5. **Arabic**: مرحبا (Marhaban)
6. **Portuguese**: Olá
7. **Finnish**: Hei
8. **Maori**: Kia ora
9. **Russian**: Привет (Privet)
10. **Hawaiian**: Aloha

# Error handling Test Blocks
###### directly copied from claude since i aint writing allat big bro :wilted_rose:

In [261]:
from unittest.mock import MagicMock

class FakeChunk:
    def __init__(self, text):
        self.text = text

def make_client(stream_side_effect):
    mock_client = MagicMock()
    mock_client.models.generate_content_stream = MagicMock(side_effect=stream_side_effect)
    return mock_client

def make_api_error(code, message="", status=None):
    """
    Builds an errors.APIError instance without calling its real __init__
    (whose signature varies by SDK version). We only need it to satisfy
    `except errors.APIError as e` and expose .code / .message / .status,
    since that's all execute_prompt reads off it.
    """
    e = errors.APIError.__new__(errors.APIError)
    e.code = code
    e.message = message
    e.status = status
    return e

random.uniform = lambda a, b: 1.0

def patch_sleep():
    sleep_mock = MagicMock()
    globals()['time'].sleep = sleep_mock
    return sleep_mock

In [262]:
sleep_mock = patch_sleep()
client = make_client([[FakeChunk("hello"), FakeChunk("world")]])

result = list(execute_prompt("gemini-test", "hi"))

assert [c.text for c in result] == ["hello", "world"]
assert client.models.generate_content_stream.call_count == 1
sleep_mock.assert_not_called()
print("PASSED: happy path")

Sending initial request...
PASSED: happy path


happy path:

In [263]:
sleep_mock = patch_sleep()
err = make_api_error(code=503, message="Service unavailable", status="UNAVAILABLE")
client = make_client([err, [FakeChunk("recovered")]])

result = list(execute_prompt("gemini-test", "hi"))

assert [c.text for c in result] == ["recovered"]
assert client.models.generate_content_stream.call_count == 2
sleep_mock.assert_called_once()
print("PASSED: retry then success")

Sending initial request...
 Code 503. Backing off. Waiting 1.00s...
Retry attempt 2/3...
PASSED: retry then success


 retryable error, then success:

In [264]:
sleep_mock = patch_sleep()
err = make_api_error(code=429, message="Rate limited", status="RESOURCE_EXHAUSTED")
client = make_client([err, err, err])

try:
    list(execute_prompt("gemini-test", "hi"))
    assert False, "expected APIError to be raised"
except errors.APIError as e:
    assert e is err

assert client.models.generate_content_stream.call_count == 3
assert sleep_mock.call_count == 2
print("PASSED: retry exhaustion raises")

Sending initial request...
 Code 429. Backing off. Waiting 1.00s...
Retry attempt 2/3...
 Code 429. Backing off. Waiting 2.00s...
Retry attempt 3/3...
Max attempts reached.
PASSED: retry exhaustion raises


retryable error, all attempts exhausted:

In [265]:
sleep_mock = patch_sleep()
Retry_Config.max_attempts = 4
Retry_Config.attempts = 4
Retry_Config.base_delay = 1.0
Retry_Config.max_delay = 3.0
Retry_Config.retry_status_codes = {503}

err = make_api_error(code=503, message="unavailable")
client = make_client([err, err, err, [FakeChunk("ok")]])

list(execute_prompt("gemini-test", "hi"))

delays = [call.args[0] for call in sleep_mock.call_args_list]
assert delays == [1.0, 2.0, 3.0], delays  # unclamped would be 1,2,4 -> clamped to 3
print("PASSED: backoff growth + clamp")

# reset config for later cells
Retry_Config.max_attempts = 3
Retry_Config.attempts = 3
Retry_Config.max_delay = 8.0
Retry_Config.retry_status_codes = {429, 500, 503}

Sending initial request...
 Code 503. Backing off. Waiting 1.00s...
Retry attempt 2/4...
 Code 503. Backing off. Waiting 2.00s...
Retry attempt 3/4...
 Code 503. Backing off. Waiting 3.00s...
Retry attempt 4/4...
PASSED: backoff growth + clamp


backoff grows and clamps to max_delay:

In [266]:
sleep_mock = patch_sleep()
err = make_api_error(code=404, message="not found", status="NOT_FOUND")
client = make_client([err])

try:
    list(execute_prompt("nonexistent-model", "hi"))
    assert False
except errors.APIError:
    pass

assert client.models.generate_content_stream.call_count == 1
sleep_mock.assert_not_called()
print("PASSED: 404 non-retryable")

Sending initial request...
Model Not Found [404]: 'nonexistent-model' does not exist or is deprecated.
PASSED: 404 non-retryable


non-retryable 404:

In [267]:
for code, status in [(401, None), (403, None), (999, "PERMISSION_DENIED")]:
    err = make_api_error(code=code, message="denied", status=status)
    client = make_client([err])
    try:
        list(execute_prompt("gemini-test", "hi"))
        assert False
    except errors.APIError:
        pass
print("PASSED: auth failure cases")

Sending initial request...
Authentication Failed [401]: Check your API key, project quotas, or permissions.
Sending initial request...
Authentication Failed [403]: Check your API key, project quotas, or permissions.
Sending initial request...
Authentication Failed [999]: Check your API key, project quotas, or permissions.
PASSED: auth failure cases


non-retryable 401/403 and PERMISSION_DENIED:

In [268]:
for code, status in [(400, None), (999, "INVALID_ARGUMENT")]:
    err = make_api_error(code=code, message="bad request", status=status)
    client = make_client([err])
    try:
        list(execute_prompt("gemini-test", "hi"))
        assert False
    except errors.APIError:
        pass
print("PASSED: invalid argument cases")

Sending initial request...
Invalid Argument [400]: Malformed structure or prompt content limits exceeded.
Sending initial request...
Invalid Argument [400]: Malformed structure or prompt content limits exceeded.
PASSED: invalid argument cases


non-retryable unknown code (generic fallback message):

In [269]:
err = make_api_error(code=418, message="I'm a teapot", status="UNKNOWN_STATUS")
client = make_client([err])

try:
    list(execute_prompt("gemini-test", "hi"))
    assert False
except errors.APIError:
    pass
print("PASSED: unknown code generic message")

Sending initial request...
Non-retryable error [UNKNOWN_STATUS] (418): I'm a teapot
PASSED: unknown code generic message
